# 09. LightGBM 과거 시점 재평가

현재 Valid/Test와 겹치지 않는 두 과거 시점에서 **설정을 고정한 LightGBM**과 4주 동일 요일 기준선을 비교합니다. 각 시점은 Valid 7일 → 간격 7일 → Test 7일이며, 해당 Valid 시작보다 앞선 정답으로만 모델을 학습합니다. Valid로 학습 종료 시점을 정한 모델을 그대로 Test에 씁니다. 전체 이력 Feature 캐시를 재사용하지만 GPU 학습은 각 시점마다 다시 필요합니다.

이 실험은 이미 현재 Test 결과를 본 후 설계한 **회고적 강건성 점검**입니다. 독립적인 미래 배포 성능 보증으로 해석하지 않습니다. 실행 시간과 GPU 메모리를 확인한 뒤 필요한 시점만 실행하세요.

## Colab 준비

별도의 00 노트북으로 저장소를 My Drive에 준비합니다. 기존 Feature 캐시가 없으면 05의 생성 셀을 먼저 실행하세요.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
%cd /content/drive/MyDrive/retail-demand-forecasting

In [ ]:
%pip install -r requirements.txt

## Library

In [ ]:
import sys
from dataclasses import replace
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import koreanize_matplotlib

REPO = Path("/content/drive/MyDrive/retail-demand-forecasting")
assert (REPO / "src/retail_forecast/__init__.py").is_file(), REPO
sys.path.insert(0, str(REPO / "src"))

from retail_forecast.config import load_config
from retail_forecast.metrics import regression_metrics
from retail_forecast.pipeline import feature_cache_path, run_time_split_experiment

sns.set_theme(style="whitegrid", font="NanumGothic")
plt.rcParams["axes.unicode_minus"] = False

## Utils

In [ ]:
def fold_config(last_origin):
    config = load_config(REPO / "configs/full_lightgbm_gpu.toml")
    run_name = f"lightgbm_backtest_{last_origin:%Y%m%d}"
    return replace(
        config,
        model_params={**config.model_params, "n_estimators": 600,
                      "max_depth": -1, "early_stopping_rounds": 100},
        model_path=REPO / "models" / run_name,
        output_path=REPO / "outputs" / run_name,
    )


def weekly_frame(predictions):
    return predictions.groupby(["date", "store_nbr", "family"], observed=True)[
        ["target_sales", "prediction", "sales_same_weekday_4w_mean"]
    ].sum(min_count=7).reset_index()


def weekly_scores(valid_predictions, test_predictions):
    valid = weekly_frame(valid_predictions)
    test = weekly_frame(test_predictions)
    cutoff = valid["sales_same_weekday_4w_mean"].quantile(1 / 3)
    low_valid = valid.loc[valid["sales_same_weekday_4w_mean"].le(cutoff)]
    model_error = (low_valid["prediction"] - low_valid["target_sales"]).abs().sum()
    baseline_error = (low_valid["sales_same_weekday_4w_mean"] - low_valid["target_sales"]).abs().sum()
    use_fallback = bool(model_error > baseline_error)
    rows = []
    for phase, weekly in [("Valid", valid), ("Test", test)]:
        low = weekly["sales_same_weekday_4w_mean"].le(cutoff) & use_fallback
        forecasts = {
            "LightGBM": weekly["prediction"],
            "최근 4주 동일 요일 평균": weekly["sales_same_weekday_4w_mean"],
            "Valid 선택 규칙": weekly["prediction"].where(
                ~low, weekly["sales_same_weekday_4w_mean"]
            ),
        }
        for name, forecast in forecasts.items():
            rows.append({"phase": phase, "model": name, "rows": len(weekly),
                         "low_demand_cutoff": cutoff, "use_fallback": use_fallback,
                         **regression_metrics(weekly["target_sales"], forecast)})
    return pd.DataFrame(rows)

## 실행 시점

두 평가 구간은 현재 Valid(2017-07-19~25)와 Test(2017-08-02~08)에 앞서며 서로 겹치지 않습니다. 아래 날짜는 각 Test의 **마지막 예측 기준일**입니다. 한 번에 하나씩 실행하고 결과를 살펴보세요. `RUN_BACKTEST = False`인 상태에서는 재학습하지 않습니다.

In [ ]:
BACKTEST_END_DATES = [pd.Timestamp("2017-06-20"), pd.Timestamp("2017-07-11")]
RUN_BACKTEST = False

cache_config = fold_config(BACKTEST_END_DATES[0])
cache_path = feature_cache_path(cache_config)
assert cache_path.exists(), f"전체 이력 Feature 캐시가 없습니다: {cache_path}"
display(pd.DataFrame({
    "test_last_origin": BACKTEST_END_DATES,
    "valid_start": [date - pd.Timedelta(days=20) for date in BACKTEST_END_DATES],
    "valid_end": [date - pd.Timedelta(days=14) for date in BACKTEST_END_DATES],
    "test_start": [date - pd.Timedelta(days=6) for date in BACKTEST_END_DATES],
}))

## 과거 시점 GPU 학습과 비교

한 시점당 전체 Feature 캐시를 읽고 LightGBM을 한 번 학습합니다. 기존 05·07 결과는 덮어쓰지 않습니다. 먼저 한 시점만 선택해 실행 시간을 확인한 후 나머지를 실행하세요. 각 시점 Test의 7일 합계 WAPE·Bias와 기준선 대비 개선을 별도 CSV로 저장합니다.

In [ ]:
if RUN_BACKTEST:
    summaries = []
    for last_origin in BACKTEST_END_DATES:
        config = fold_config(last_origin)
        run_time_split_experiment(config, progress=True, last_origin=last_origin)
        valid_predictions = pd.read_csv(config.output_path / "validation_predictions_lightgbm.csv")
        test_predictions = pd.read_csv(config.output_path / "test_predictions_lightgbm.csv")
        scores = weekly_scores(valid_predictions, test_predictions)
        scores.insert(0, "test_last_origin", last_origin)
        scores.to_csv(config.output_path / "test_weekly_baseline_comparison.csv", index=False)
        summaries.append(scores)
        display(scores)

    backtest_summary = pd.concat(summaries, ignore_index=True)
    backtest_summary.to_csv(REPO / "outputs/lightgbm_backtest_summary.csv", index=False)
    display(backtest_summary)

## 읽는 법

각 시점에서 LightGBM의 7일 WAPE가 기준선보다 낮은지, Bias가 한쪽으로 지속되는지 확인합니다. 낮은 수요의 대체 경계와 사용 여부는 **해당 시점 Valid만으로** 정합니다. 그 뒤 Test에서 모델·기준선·대체 규칙을 함께 비교합니다. 이미 현재 Test를 본 뒤 설계한 규칙인 만큼, 과거 시점의 반복 개선은 회고적 강건성 자료로 보고합니다.